In [1]:


import os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import resnet50
from sklearn.model_selection import train_test_split

torch.backends.quantized.engine = "fbgemm"
DEVICE = torch.device("cpu")

# dataset 
transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
NUM_CLASSES = len(dataset.classes)
print(f"Total images: {len(dataset)}, Classes: {NUM_CLASSES}")

# stratified split
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for cid in np.unique(targets):
    idx = np.where(targets == cid)[0]
    tr, tmp = train_test_split(idx, test_size=0.15, random_state=42, shuffle=True)
    va, te = train_test_split(tmp, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(tr); val_idx.extend(va); test_idx.extend(te)

train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

batch_size = 128
nw = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=nw, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=nw, pin_memory=True)
print("DataLoaders ready")

# model + weights 
state = torch.load("resnet50_weights.pth", map_location="cpu")
if any(k.startswith("_orig_mod.") for k in state.keys()):
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}

model_fp32 = resnet50(weights=None)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, NUM_CLASSES)
model_fp32.load_state_dict(state, strict=True)
model_fp32.eval().to(DEVICE)
print("FP32 ResNet-50 loaded")

# PoT quant function
def power_of_two_quantize(weight: torch.Tensor, n_bits=8):
    w = weight.clone()
    w_min, w_max = w.min(), w.max()
    scale = (w_max - w_min) / (2 ** n_bits - 1 + 1e-12)
    scale = torch.clamp(scale, min=1e-8)
    pow2_scale = 2 ** torch.round(torch.log2(scale))
    zp = torch.round(-w_min / pow2_scale)
    w_q = torch.round(w / pow2_scale + zp)
    w_q = torch.clamp(w_q, 0, 2 ** n_bits - 1)
    q_w = (w_q - zp) * pow2_scale
    return q_w

# Apply PoT (weights only) 
model_pot = resnet50(weights=None)
model_pot.fc = nn.Linear(model_pot.fc.in_features, NUM_CLASSES)
model_pot.load_state_dict(state, strict=True)
for name, p in model_pot.named_parameters():
    if "weight" in name:
        with torch.no_grad():
            p.copy_(power_of_two_quantize(p, n_bits=8))
model_pot.eval().to(DEVICE)
print("Applied Power-of-Two Quantization")

# eval / size / runtime 
def evaluate(model, loader):
    model.eval()
    corr, tot = 0, 0
    with torch.inference_mode():
        for x, y in loader:
            out = model(x.to("cpu"))
            pred = out.argmax(1).cpu()
            corr += (pred == y).sum().item()
            tot  += y.size(0)
    return 100.0 * corr / tot

def get_model_size(model, fname="__tmp__.pth"):
    torch.save(model.state_dict(), fname)
    sz = os.path.getsize(fname) / 1e6
    os.remove(fname)
    return sz

def benchmark(model, loader, num_batches=50):
    model.eval()
    t0 = time.time()
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            if i >= num_batches: break
            _ = model(x.to("cpu"))
    t1 = time.time()
    imgs = num_batches * loader.batch_size
    return (t1 - t0) / num_batches * 1000.0, (t1 - t0) / imgs * 1000.0

acc_fp32 = evaluate(model_fp32, test_loader)
acc_pot  = evaluate(model_pot,  test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"Power-of-Two Accuracy: {acc_pot:.2f}%")

fp32_sz = get_model_size(model_fp32)
pot_sz  = get_model_size(model_pot)
print(f"FP32 size: {fp32_sz:.2f} MB | Power-of-Two size: {pot_sz:.2f} MB")

b_ms_fp32, i_ms_fp32 = benchmark(model_fp32, test_loader)
b_ms_pot,  i_ms_pot  = benchmark(model_pot,  test_loader)
print(f"FP32 Inference: {b_ms_fp32:.2f} ms/batch | {i_ms_fp32:.2f} ms/image")
print(f"Power-of-Two Inference: {b_ms_pot:.2f} ms/batch | {i_ms_pot:.2f} ms/image")




Total images: 100000, Classes: 200


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


DataLoaders ready
FP32 ResNet-50 loaded
Applied Power-of-Two Quantization
FP32 Accuracy: 73.90%
Power-of-Two Accuracy: 70.14%
FP32 size: 95.98 MB | Power-of-Two size: 95.98 MB
FP32 Inference: 12527.62 ms/batch | 97.87 ms/image
Power-of-Two Inference: 12276.23 ms/batch | 95.91 ms/image
